
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# MLflow 그리고 에이전트 개발

## 서론

회수 에이전트 구축은 표준 모델 학습과 다릅니다. 이것은 사용자 쿼리, 임베딩 모델, 벡터 데이터베이스, 대형 언어 모델 간의 동적 상호작용을 조율하는 것을 포함합니다. 이 강의는 **MLflow 3.0+** 가 이러한 에이전트를 개발, 디버깅, 거버넌스하는 데 필요한 인프라를 어떻게 제공하는지 탐구합니다. 우리는 단순한 기록을 넘어 Databricks Unity Catalog를 사용하여 검색 단계의 깊은 추적성과 에이전트 산출물의 거버넌스를 탐구할 것입니다.

## 수업 목표

* 핵심 **MLflow 구성** 요소와 에이전트 개발에서의 구체적 역할을 식별하세요.  
* 프롬프트와 리트리버 설정의 변동을 추적할 수 있도록 **실험**을 설정하세요.  
* 표준 모델 맛과 **생성형 AI 전용 유형**(LangChain, PyFunc)을 구분하세요.  
* **MLflow 추적**을 활용하여 특정 검색 실패(예: 빈 검색 결과, 높은 지연 시간)를 진단합니다.  
* **Unity Catalog Model Registry**를 사용하여 회수 에이전트를 등록하고 관리합니다.

## A. 에이전트를 위한 MLflow의 기초

MLflow는 머신 러닝 라이프사이클을 관리하기 위해 설계된 오픈 소스 플랫폼입니다. 에이전트 개발 맥락에서 MLflow는 모든 구성, 코드 버전, 실행 추적에 대한 중앙 시스템 역할을 합니다.

그 가치를 이해하기 위해 "No MLflow" 시나리오를 고려해 보십시오: 개발자들은 복잡한 체인을 디버깅하기 위해 종종 산발된 print() 문이나 기본 로그에 의존합니다. 이 접근법은 특정 쿼리가 왜 실패했는지 이해해야 할 때 실패합니다—그것이 AI Search의 타임아웃인지, 임베딩 모델에 잘못된 쿼리인지, 아니면 추론 오류인지? 구조화된 추적 시스템이 없으면 이러한 중간 고장과 특정 구성 변경을 연관짓는 것은 거의 불가능해집니다.

**MLflow**를 사용하면 검색 매개변수부터 최종 생성까지 에이전트의 모든 행동 측면이 체계적으로 기록됩니다. 이렇게 하면 "어떤 구성이 이 고품질 응답을 만들어냈는가?"라는 질문에 확신을 가지고 답할 수 있습니다.

### A1. MLflow의 구성 요소

구체적인 Workflows에 들어가기 전에 플랫폼의 아키텍처 기둥을 이해하는 것이 필수적입니다. MLflow는 단일 도구가 아니라, 에이전트 라이프사이클의 다양한 단계를 처리하는 통합 구성 요소들로, 코드 첫 줄부터 최종 생산 거버넌스까지 모두 처리합니다.

* **MLflow 추적:** 매개변수, 코드 버전, 메트릭, 출력 파일 기록을 위한 API와 UI. 검색 에이전트의 경우, 추적 시스템 프롬프트와 검색기 구성 등이 포함됩니다.  
* **MLflow 추적:** 특정 검색 도구 호출을 디버깅하는 데 필수적인 에이전트의 계층적 실행 흐름을 포착하는 전용 관측성 기능입니다.  
* **MLflow 모델:** 라이브러리와 상관없이 다양한 하위 도구(예: 실시간 서빙)에서 사용할 수 있는 모델 패키징의 표준 형식입니다.  
* **MLflow Model Registry:** 모델 수명주기 관리, 버전 관리, 스테이지에서 프로덕션으로의 전환 등 단계별 협업을 위한 중앙 집중식 저장소입니다.

### A2. 실험과 실행

구성 요소를 이해한 후, 어떤 개발 주기든 첫 단계는 반복 작업을 조직하는 것입니다. 회수 에이전트를 테스트할 때, 20가지의 다양한 시스템 프롬프트나 청킹 전략을 시도할 수 있지만, 구조가 없으면 금세 혼란스러워집니다.

**실험**은 “고객 지원 검색 에이전트”와 같은 특정 프로젝트의 주요 논리적 컨테이너 역할을 합니다. 실험 내에서 개별 **실행**은 특정 시점의 에이전트 상태를 포착합니다. MLflow는 각 실행에 대한 **추론 엔진 구성**을 기록함으로써 재현성 문제를 해결합니다:

* **시스템 프롬프트:** 에이전트의 페르소나를 정의하는 구체적인 지침(예: "당신은 검색된 맥락에만 기반해 응답하는 도움이 되는 비서입니다").  
* **모델 구성:** 온도 및 `max_tokens`와 같은 매개변수.  
* **리트리버 설정:** 추출해야 할 청크 수(k)나 벡터 유사성의 필터링 임계값과 같은 중요한 매개변수들.

개발자들은 이러한 `mlflow.set_experiment()` 실행이 저장되는 워크스페이스 위치를 정의하여 검색 로직의 반복을 조직합니다.

### A3. 모델 플레이버와 래퍼

실험을 기록하고 성공적인 구성을 찾은 후에는 해당 에이전트를 배포할 수 있는 패키징 방법이 필요합니다. Python 스크립트를 단순히 저장해두고 그것이 의존성, 환경, 특정 로딩 로직 없이 프로덕션에서 작동하기를 기대할 수는 없습니다.

**Model Flavor**는 MLflow가 사용자가 이러한 의존성을 수동으로 처리하지 않고도 모델을 저장, 불러오고 서비스를 제공할 수 있게 해주는 통합입니다.

* **Native GenAI Flavors:** MLflow는 **LangChain**(mlflow.langchain)와 **OpenAI** 같은 라이브러리에 대한 네이티브 지원을 포함합니다. 이 플레버들은 검색 체인과 그 구성 요소의 직렬화를 자동으로 처리합니다.  
* **PyFunc Flavor:** 생산용 검색 에이전트의 경우, 특정 재랭킹 단계나 동적 필터 적용 같은 맞춤형 로직이 필요하며, 네이티브 플레버에서는 커버하지 않을 수 있습니다. Python 함수(PyFunc) 특성은 그것이 predict() 메서드를 노출하는 경우에 한해, 임의의 Python 코드를 모델로 감쌀 수 있게 해줍니다.

**참고:** PyFunc를 검색 에이전트로 사용할 때는 사용자 지정 검색 코드와 필요한 구성 파일이 기록된 산출물에 포함되었는지 확인하세요.


## B. 관찰 가능성 및 추적

이제 패키지 에이전트를 확보했으니, 새로운 도전에 직면했습니다: **그것이 그렇게 행동하는 이유를 이해한다**. 정확성만 확인하는 전통적인 모델과 달리, 검색 에이전트는 사용자, 벡터 데이터베이스, LLM 간의 상호작용을 '블랙박스'로 구성하여 표준 디버깅 방법으로는 효과가 없습니다.

### B1. 추적의 필요성

사용자가 "원격 근무에 대한 정책이 무엇인가요?"라고 묻고 상담원이 "모르겠습니다"라고 답하면, 단순한 로그 텍스트로는 이유를 알 수 없습니다. 검색 도구가 문서를 찾지 못했나요? 검색이 느리고 시간이 지나갔나요? 아니면 LLM이 검색된 맥락을 무시한 걸까요? **MLflow 트레이싱**은 체인의 모든 단계의 입력과 출력을 기록하여 이 실행 그래프를 고충실도로 가시화합니다.

### B2. 트레이스와 스팬

<!-- <img src="../Includes/images/04-mlflow-tracing-ui.png" alt="MLflow tracing UI" /> -->
![04-mlflow-tracing-ui](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/04-mlflow-tracing-ui.png)

*그림 1. 이 다이어그램은 MLflow의 추적 UI를 보여줍니다.* 

MLflow 추적은 **트레이스**와 **스팬**를 사용하여 실행 흐름을 시각화합니다.

* **트레이스:** 사용자의 초기 질문부터 최종 답변까지의 전체 요청 수명 주기를 나타냅니다.  
* **스팬:** 개별 작업 단위를 나타냅니다. 회수 에이전트의 경우, 보통 "query_embedding", "retrieval_tool", "context_generation"에 대한 구체적인 구간을 볼 수 있습니다.

추적은 지원되는 라이브러리(예: `mlflow.langchain.autolog()` **Auto-logging**)를 통해 활성화할 수 있으며, 사용자 지정 검색 기능을 위해 `@mlflow.trace` 데코레이터를 사용하는 **수동 계측**을 통해 활성화할 수 있습니다.

### B3. 검색 실패 진단

추적의 주요 가치는 검색 도구의 특정 고장 모드를 디버깅하는 데 있습니다. 추적은 개발자가 표준 로그에서 보이지 않는 문제를 정확히 찾아낼 수 있게 해줍니다:

1. **비어 있거나 관련 없는 검색:** **Retriever Span**의 출력을 검사하면 벡터 데이터베이스에서 반환된 청크를 정확히 확인할 수 있습니다. 만약 span 출력이 비어 있거나 좋은 쿼리에도 불구하고 관련 없는 텍스트가 있다면, 문제는 LLM이 아니라 임베딩 모델이나 청킹 전략에 있다는 것을 알 수 있습니다.  
2. **AI Search에서의 지연 시간:** 스팬은 **지연 시간**(지속 시간)을 캡처합니다. 에이전트가 느리다면, 트레이스 워터폴은 `vector_search` 스팬이 4초였고 LLM 생성은 500ms에 불과하다는 것을 알 수 있습니다. 이로 인해 최적화 노력은 모델이 아닌 데이터베이스 쿼리에 집중됩니다.  
3. **문맥에도 불구하고 환각:** 추적 결과 리트리버 스팬이 올바른 문서를 반환했음에도 LLM 스팬 출력이 이를 무시하는 경우, 추론 실패를 확인한 것입니다. 이는 제공된 컨텍스트에 대한 엄격한 준수를 강제하기 위해 시스템 프롬프트를 개선할 필요가 있음을 시사합니다.


## C. Unity Catalog를 통한 거버넌스

기능하고 디버깅된 에이전트가 있으면, 우리는 마지막 난관에 직면합니다: 생산 거버넌스. 개발자가 검증 없이 코드를 직접 생산 엔드포인트에 Push 하는 것을 허용할 수 없으며, 기본 데이터에 대한 거버넌스 없는 접근도 허용할 수 없으므로 강력한 레지스트리 시스템이 필수입니다.

### C1. Unity Catalog Model Registry

기업 환경에 배포되는 에이전트는 엄격한 거버넌스와 감독이 필요합니다. **Unity Catalog (UC)** 는 이 자산들의 중앙 등록 역할을 합니다. 기존 Workspace Model Registry와 달리, UC는 데이터와 AI 자산 간의 접근 제어를 통합하는 3단계 네임스페이스`catalog.schema.model`를 제공합니다.

* **접근 제어:** 등록된 에이전트에 대한 권한(`SELECT`, `EXECUTE`)은 기본 AI Search 테이블과 마찬가지로 관리할 수 있습니다.  
* **리니지(Lineage)** UC는 에이전트가 어떤 데이터 테이블(AI Search 인덱스를 통해)을 사용했는지 추적하여 원시 문서에서 배포된 에이전트까지 종단 간 리니지를 제공합니다.

### C2. 로그 기록 및 에이전트 등록

에이전트를 거버넌스하는 워크플로는 특정 서명으로 모델을 기록한 후 모델을 등록하는 과정을 포함합니다.

1. **모델 서명 정의하기:** 에이전트는 일반적으로 문자열 입력이나 채팅 기록 목록을 받습니다. 이 입출력 스키마를 `mlflow.models.ModelSignature`를 사용하여 정의하여 서빙 엔드포인트가 요청을 올바르게 검증하도록 해야 합니다.  
2. **Log 모델:** `mlflow.langchain.log_model`를 사용 (또는 적절한 스타일). 최고의 방법은 UI가 작동하는 테스트 위젯을 생성할 수 있도록 하는 **입력 예시**를 포함하는 것입니다.  
3. **등록:** 실험에 로그되면 모델 버전은 다음을 사용하여 Unity Catalog에 등록됩니다:  
   `mlflow.register_model("runs:/<run_id>/model", "catalog.schema.retrieval_agent")`

**참고:** **검색 도구** 자체(Unity Catalog 함수로 정의된 경우)도 동일한 카탈로그 구조 내에서 관리되어야 일관된 보안 경계를 유지할 수 있습니다.


<!-- <img src="../Includes/images/04-model-registery.png" alt="Model Registery in UC" /> -->

![04-model-registery](https://files.training.databricks.com/binder/prod_main/building-retrieval-agents-on-databricks-ko_kr-1.0.1/images/04-model-registery.png)

*그림 2. 이 도표는 UC 모델 레지스트리 인터페이스를 보여줍니다.* 


*참고:* [Manage models in Unity Catalog Documentation](https://docs.databricks.com/aws/en/machine-learning/manage-model-lifecycle)


## D. 요약

이 수업에서는 MLflow의 검색 중심 Workflows에 맞게 어떻게 적응했는지 설명했습니다. 우리는 **MLflow**의 핵심 구성 요소와 **실험**이 리트리버와 프롬프트의 특정 구성을 어떻게 포착하는지를 정의했습니다. 우리는 검색 실패(검색 결과가 좋지 않음)와 추론 실패(예: 환각)를 구분하는 데 중요한 도구로서 **MLflow 추적**을 탐구했습니다. 마지막으로, 우리는 **Unity Catalog**가 이 에이전트들의 버전 관리를 위한 거버넌스 레지스트리를 제공하는 역할에 대해 다뤘습니다.

**핵심 내용:**

1. **추적은 검색에 필수적입니다:** 중간 검색 범위 출력을 보지 않고는 에이전트가 "모르겠다"고 말한 이유를 효과적으로 디버깅할 수 없습니다.  
2. **Custom Logic용 PyFunc:** 복잡한 검색 전략은 종종 pyfunc 래퍼가 사용자 정의 재순위 또는 필터링 논리를 캡슐화해야 합니다.  
3. **Unity Catalog를 통한 거버넌스:** 에이전트는 Unity Catalog(catalog.schema.model)에 등록되어 원본 문서로의 계보와 안전한 접근 제어를 보장해야 합니다.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>